# C1 · Temporal leakage: random K-fold on a time series

**Type**: Break the model · **Level**: ●●○ independent · **Time**: about 40 minutes · **core path ★**

## Objective

Show that on a time series a random K-fold gives an R² close to 1 for a model that actually cannot predict the
future, compare it with `TimeSeriesSplit` and with a true out-of-sample forecast, and compare the model with a seasonal baseline.

## Dataset

`ds.time_series()`: 600 months. `y` = random walk (trend) + annual seasonality + noise. The features are `t` (month
index) and the season (`s`, `c`). A `RandomForestRegressor` with these features can memorize "what `y` is around month t"
but does not know what the trend will do in the following months.

## Educational purpose

Get a hands-on feel for why the splitter must mirror the real use: predicting the future. In a random K-fold every validation month
has its adjacent months in the training set and the model **interpolates**; in production it will have to **extrapolate**.

## Task

1. Compute `r2_kfold` with `cross_val_score(rf, X, y, cv=KFold(5, shuffle=True, random_state=0), scoring="r2").mean()`.
2. Compute `r2_ts` with `TimeSeriesSplit(5)`.
3. Compute `r2_future`: train `rf` on the first 480 months and evaluate the R² on the last 120 (the real use situation).
4. Compute `r2_seasonal`: R² on the last 120 months of the naive forecast "same month of the previous year" (`y.shift(12)`), which does not require any model.
5. Compute `kfold_distance = |r2_kfold − r2_future|` and `ts_distance = |r2_ts − r2_future|`.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold, TimeSeriesSplit, cross_val_score

s = ds.time_series()
X, y = s[["t", "s", "c"]], s["y"]
rf = RandomForestRegressor(100, random_state=0)
cut = 480

## Your work

In [ ]:
r2_kfold = ...
r2_ts = ...
print(f"R2 with random KFold: {r2_kfold:.3f} | with TimeSeriesSplit: {r2_ts:.3f}")

In [ ]:
r2_future = ...
r2_seasonal = ...
kfold_distance = ...
ts_distance = ...
print(f"true R2 on the last 120 months: {r2_future:.3f} | forecast 'same month of the previous year': {r2_seasonal:.3f}")

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='`r2_future`: `rf.fit(X.iloc[:cut], y.iloc[:cut]); r2_score(y.iloc[cut:], rf.predict(X.iloc[cut:]))`. Baseline: `r2_score(y.iloc[cut:], y.shift(12).iloc[cut:])` (the only rows with NaN are in the first 12 months, not in the last block).'):
    check(r2_kfold > 0.9, "the random K-fold gives a very high R2 (above 0.9)", "with random KFold the model interpolates between adjacent months")
    check(r2_ts < 0.6, "with TimeSeriesSplit the R2 collapses (below 0.6)", "TimeSeriesSplit trains on the past and evaluates on the future")
    check(close(kfold_distance, abs(r2_kfold - r2_future)) and close(ts_distance, abs(r2_ts - r2_future)), "distances computed correctly",
          "distance = |estimated R2 - true R2 on the future|")
    check(ts_distance < kfold_distance / 2, "the temporal estimate is much closer to the truth than the random K-fold one",
          "TimeSeriesSplit must get much closer to the true R2 on the future than the random KFold")
    check(r2_seasonal > r2_future + 0.3, "the baseline 'same month of the previous year' beats the random forest on the future (by more than 0.3)",
          "the forest with t as a feature cannot extrapolate the trend: the seasonal baseline requires no model")
done()

---
The commented solution is in `solutions/C_break_the_model/C1_temporal_leakage_kfold_on_series.ipynb`. Open it after you pass the check, or after you have really tried.